In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Global Settings
pd.set_option("display.max_columns", None)

# 0. Data Loading

In [2]:
curr_path = os.getcwd()
csv_path = curr_path.replace("data_analysis", "datasets/NF-CSE-CIC-IDS2018-v3/NF-CSE-CIC-IDS2018-v3.csv")
timestamp_cols = {"FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS"}
chunks = []
for chunk in pd.read_csv(csv_path, chunksize=500_000):
    for col in chunk.select_dtypes(include=["float64"]).columns:
        chunk[col] = chunk[col].astype(np.float32)
    for col in chunk.select_dtypes(include=["int64"]).columns:
        if col in timestamp_cols:
            continue
        elif col == "Label":
            chunk[col] = chunk[col].astype(np.int8)
        else:
            chunk[col] = chunk[col].astype(np.int32)
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)
del chunks

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20115529 entries, 0 to 20115528
Data columns (total 55 columns):
 #   Column                       Dtype
---  ------                       -----
 0   FLOW_START_MILLISECONDS      int64
 1   FLOW_END_MILLISECONDS        int64
 2   IPV4_SRC_ADDR                str  
 3   L4_SRC_PORT                  int32
 4   IPV4_DST_ADDR                str  
 5   L4_DST_PORT                  int32
 6   PROTOCOL                     int32
 7   L7_PROTO                     int32
 8   IN_BYTES                     int32
 9   IN_PKTS                      int32
 10  OUT_BYTES                    int32
 11  OUT_PKTS                     int32
 12  TCP_FLAGS                    int32
 13  CLIENT_TCP_FLAGS             int32
 14  SERVER_TCP_FLAGS             int32
 15  FLOW_DURATION_MILLISECONDS   int32
 16  DURATION_IN                  int32
 17  DURATION_OUT                 int32
 18  MIN_TTL                      int32
 19  MAX_TTL                      int32
 20  LONGEST_FLO

# 1. Data Overview

In [3]:
df.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1518611287705,1518611287705,172.31.0.2,53,172.31.66.58,63593,17,5,156,1,0,0,0,0,0,1,0,0,0,0,156,156,0,156,0,1,0,0,0,0,1248000,0,0,1,0,0,0,0,0,0,0,45856,1,60,0,0,0,0,0,0,0,0,0,0,Benign
1,1518611287743,1518611290747,172.31.66.58,56163,239.255.255.250,1900,17,12,805,5,0,0,0,0,0,3003,3003,0,1,1,161,161,0,161,0,0,0,0,0,0,2143,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,2793,750,1182,0,0,0,0,0,Benign
2,1518611288143,1518611300202,172.31.66.46,62388,239.255.255.250,1900,17,12,1288,8,0,0,0,0,0,12058,12058,0,1,1,161,161,0,161,0,0,0,0,0,0,854,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,3016,1722,1436,0,0,0,0,0,Benign
3,1518611288165,1518611336194,0.0.0.0,546,0.0.0.0,547,17,103,393,3,0,0,0,0,0,48029,48029,0,1,1,131,131,0,131,0,0,0,0,0,0,65,0,0,3,0,0,0,0,0,0,0,0,0,0,0,16014,32014,24014,8000,0,0,0,0,0,Benign
4,1518611288175,1518611288176,172.31.66.46,49187,169.254.169.254,80,6,7,373,5,700,5,27,27,27,1,0,0,128,128,528,40,40,528,700,373,0,0,0,0,1492000,2800000,8,1,0,1,0,8192,17922,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Benign


In [4]:
df.tail()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
20115524,1519334572956,1519334662435,52.14.77.172,1024,172.31.69.28,500,17,79,3168,6,0,0,0,0,0,89479,89479,0,62,62,528,528,0,528,0,0,0,0,0,0,283,0,0,0,0,6,0,0,0,0,0,0,0,0,0,4000,41989,17895,13724,0,0,0,0,1,Brute_Force_-XSS
20115525,1519334689509,1519334778989,13.58.42.57,1028,172.31.69.28,500,17,79,3168,6,0,0,0,0,0,89479,89479,0,62,62,528,528,0,528,0,0,0,0,0,0,283,0,0,0,0,6,0,0,0,0,0,0,0,0,0,4000,41990,17895,13725,0,0,0,0,1,Brute_Force_-XSS
20115526,1519334738019,1519334827498,52.14.77.172,1024,172.31.69.28,500,17,79,3168,6,0,0,0,0,0,89479,89479,0,62,62,528,528,0,528,0,0,0,0,0,0,283,0,0,0,0,6,0,0,0,0,0,0,0,0,0,4000,41990,17895,13725,0,0,0,0,1,Brute_Force_-XSS
20115527,1519334854572,1519334944052,13.58.42.57,1028,172.31.69.28,500,17,79,3168,6,0,0,0,0,0,89479,89479,0,62,62,528,528,0,528,0,0,0,0,0,0,283,0,0,0,0,6,0,0,0,0,0,0,0,0,0,4000,41990,17895,13725,0,0,0,0,1,Brute_Force_-XSS
20115528,1519334903081,1519334992561,52.14.77.172,1024,172.31.69.28,500,17,79,3168,6,0,0,0,0,0,89479,89479,0,62,62,528,528,0,528,0,0,0,0,0,0,283,0,0,0,0,6,0,0,0,0,0,0,0,0,0,4000,41990,17895,13725,0,0,0,0,1,Brute_Force_-XSS


In [6]:
df.shape

(20115529, 55)

This are the features on the dataset and their description according to the dataset documentation:

| Feature | Description |
| --- | --- |
| IPV4 SRC ADDR | IPv4 source address |
| IPV4 DST ADDR | IPv4 destination address |
| L4 SRC PORT | IPv4 source port number |
| L4 DST PORT | IPv4 destination port number |
| PROTOCOL | IP protocol identifier byte |
| L7 PROTO | Application protocol (numeric) |
| IN BYTES | Incoming number of bytes |
| OUT BYTES | Outgoing number of bytes |
| IN PKTS | Incoming number of packets |
| OUT PKTS | Outgoing number of packets |
| FLOW DURATION MILLISECONDS | Flow duration in milliseconds |
| TCP FLAGS | Cumulative of all TCP flags |
| CLIENT TCP FLAGS | Cumulative of all client TCP flags |
| SERVER TCP FLAGS | Cumulative of all server TCP flags |
| DURATION IN | Client to Server stream duration (msec) |
| DURATION OUT | Client to Server stream duration (msec) |
| MIN TTL | Min flow TTL |
| MAX TTL | Max flow TTL |
| LONGEST FLOW PKT | Longest packet (bytes) of the flow |
| SHORTEST FLOW PKT | Shortest packet (bytes) of the flow |
| MIN IP PKT LEN | Len of the smallest flow IP packet observed |
| MAX IP PKT LEN | Len of the largest flow IP packet observed |
| SRC TO DST SECOND BYTES | Src to dst Bytes/sec |
| DST TO SRC SECOND BYTES | Dst to src Bytes/sec |
| RETRANSMITTED IN BYTES | Number of retransmitted TCP flow bytes (src->dst) |
| RETRANSMITTED IN PKTS | Number of retransmitted TCP flow packets (src->dst) |
| RETRANSMITTED OUT BYTES | Number of retransmitted TCP flow bytes (dst->src) |
| RETRANSMITTED OUT PKTS | Number of retransmitted TCP flow packets (dst->src) |
| SRC TO DST AVG THROUGHPUT | Src to dst average thpt (bps) |
| DST TO SRC AVG THROUGHPUT | Dst to src average thpt (bps) |
| NUM PKTS UP TO 128 BYTES | Packets whose IP size <= 128 |
| NUM PKTS 128 TO 256 BYTES | Packets whose IP size > 128 and <= 256 |
| NUM PKTS 256 TO 512 BYTES | Packets whose IP size > 256 and <= 512 |
| NUM PKTS 512 TO 1024 BYTES | Packets whose IP size > 512 and <= 1024 |
| NUM PKTS 1024 TO 1514 BYTES | Packets whose IP size > 1024 and <= 1514 |
| TCP WIN MAX IN | Max TCP Window (src->dst) |
| TCP WIN MAX OUT | Max TCP Window (dst->src) |
| ICMP TYPE | ICMP Type * 256 + ICMP code |
| ICMP IPV4 TYPE | ICMP Type |
| DNS QUERY ID | DNS query transaction Id |
| DNS QUERY TYPE | DNS query type (e.g., 1=A, 2=NS..) |
| DNS TTL ANSWER | TTL of the first A record (if any) |
| FTP COMMAND RET CODE | FTP client command return code |
| FLOW START MILLISECONDS | Flow start timestamp in milliseconds |
| FLOW END MILLISECONDS | Flow end timestamp in milliseconds |
| SRC TO DST IAT MIN | Minimum IAT (src->dst) |
| SRC TO DST IAT MAX | Maximum IAT (src->dst) |
| SRC TO DST IAT AVG | Average IAT (src->dst) |
| SRC TO DST IAT STDDEV | Standard deviation of IAT (src->dst) |
| DST TO SRC IAT MIN | Minimum IAT (dst->src) |
| DST TO SRC IAT MAX | Maximum IAT (dst->src) |
| DST TO SRC IAT AVG | Average IAT (dst->src) |
| DST TO SRC IAT STDDEV | Standard deviation of IAT (dst->src) |

# 2. Data Quality

In [6]:
df.isnull().sum()

FLOW_START_MILLISECONDS        0
FLOW_END_MILLISECONDS          0
IPV4_SRC_ADDR                  0
L4_SRC_PORT                    0
IPV4_DST_ADDR                  0
L4_DST_PORT                    0
PROTOCOL                       0
L7_PROTO                       0
IN_BYTES                       0
IN_PKTS                        0
OUT_BYTES                      0
OUT_PKTS                       0
TCP_FLAGS                      0
CLIENT_TCP_FLAGS               0
SERVER_TCP_FLAGS               0
FLOW_DURATION_MILLISECONDS     0
DURATION_IN                    0
DURATION_OUT                   0
MIN_TTL                        0
MAX_TTL                        0
LONGEST_FLOW_PKT               0
SHORTEST_FLOW_PKT              0
MIN_IP_PKT_LEN                 0
MAX_IP_PKT_LEN                 0
SRC_TO_DST_SECOND_BYTES        0
DST_TO_SRC_SECOND_BYTES        0
RETRANSMITTED_IN_BYTES         0
RETRANSMITTED_IN_PKTS          0
RETRANSMITTED_OUT_BYTES        0
RETRANSMITTED_OUT_PKTS         0
SRC_TO_DST

In [7]:
num_duplicates = df.duplicated().sum() 
total_involved = df.duplicated(keep=False).sum()
num_unique_pairs = num_duplicates

print(f"Total rows in dataset: {len(df):,}")
print(f"Rows marked as duplicates: {num_duplicates:,}")
print(f"Total rows involved in duplication: {total_involved:,}")

Total rows in dataset: 20,115,529
Rows marked as duplicates: 628,474
Total rows involved in duplication: 1,256,674


In [ ]:
# Remove duplicates, keeping first occurrence
df_clean = df.drop_duplicates(keep="first")

print(f"After removing duplicates (keep first): {len(df_clean):,} rows")
print(f"Rows removed: {len(df) - len(df_clean):,}")

After removing duplicates (keep first): 19,487,055 rows
Rows removed: 628,474


In [ ]:
df["Attack"].unique()

<StringArray>
[                  'Benign',           'FTP-BruteForce',
           'SSH-Bruteforce',    'DoS_attacks-GoldenEye',
    'DoS_attacks-Slowloris', 'DoS_attacks-SlowHTTPTest',
         'DoS_attacks-Hulk',   'DDoS_attacks-LOIC-HTTP',
     'DDOS_attack-LOIC-UDP',         'DDOS_attack-HOIC',
         'Brute_Force_-Web',         'Brute_Force_-XSS',
            'SQL_Injection',            'Infilteration',
                      'Bot']
Length: 15, dtype: str

In [ ]:
attack_types_original = df["Attack"].value_counts()
attack_types_clean = df_clean["Attack"].value_counts()
attack_types_dif = attack_types_original - attack_types_clean
attack_types_dif_percent = round((attack_types_dif / attack_types_original) * 100, 2)
side_by_side = pd.DataFrame({"With Duplicates": attack_types_original, "Without Duplicates": attack_types_clean, "Difference (%)": attack_types_dif_percent})
print(side_by_side)

                          With Duplicates  Without Duplicates  Difference (%)
Attack                                                                       
Benign                           17514626            17392754            0.70
Bot                                207703              111460           46.34
Brute_Force_-Web                     1618                1618            0.00
Brute_Force_-XSS                      480                 460            4.17
DDOS_attack-HOIC                  1032311             1032311            0.00
DDOS_attack-LOIC-UDP                 3450                1725           50.00
DDoS_attacks-LOIC-HTTP             288589              288589            0.00
DoS_attacks-GoldenEye               61300               30650           50.00
DoS_attacks-Hulk                   100076              100076            0.00
DoS_attacks-SlowHTTPTest           105550              105550            0.00
DoS_attacks-Slowloris               36040               18020   

In [ ]:
total_rows = len(df)
attack_rows = df[df["Attack"] != "Benign"].shape[0]
benign_rows = df[df["Attack"] == "Benign"].shape[0]
attack_percentage = (attack_rows / total_rows) * 100
benign_percentage = (benign_rows / total_rows) * 100

total_rows_clean = len(df_clean)
attack_rows_clean = df_clean[df_clean["Attack"] != "Benign"].shape[0]
benign_rows_clean = df_clean[df_clean["Attack"] == "Benign"].shape[0]
attack_percentage_clean = (attack_rows_clean / total_rows_clean) * 100
benign_percentage_clean = (benign_rows_clean / total_rows_clean) * 100

comparison_table = pd.DataFrame({
    "Dataset": ["Original", "Cleaned"],
    "Total Rows": [total_rows, total_rows_clean],
    "Attack %": [attack_percentage, attack_percentage_clean],
    "Benign %": [benign_percentage, benign_percentage_clean]
})
print(comparison_table)

    Dataset  Total Rows   Attack %   Benign %
0  Original    20115529  12.929827  87.070173
1   Cleaned    19487055  10.747140  89.252860


# 3. IPs

In [8]:
top_src_ips = df["IPV4_SRC_ADDR"].value_counts().head(10)
top_dst_ips = df["IPV4_DST_ADDR"].value_counts().head(10)
unique_src_ips = df["IPV4_SRC_ADDR"].nunique()
unique_dst_ips = df["IPV4_DST_ADDR"].nunique()

print("Top 10 Source IPs in Original Dataset:")
print(top_src_ips)
print("\nTop 10 Destination IPs in Original Dataset:")
print(top_dst_ips)
print("\nUnique Source IPs and Count:")
print(unique_src_ips)
print("\nUnique Destination IPs and Count:")
print(unique_dst_ips)

Top 10 Source IPs in Original Dataset:
IPV4_SRC_ADDR
18.221.219.4      386720
13.58.98.64       188474
5.101.40.105      138651
18.218.229.235    133850
18.216.200.189    133776
18.219.9.1        132968
18.218.55.126     132355
52.14.136.135     132322
18.218.11.51      132297
18.216.24.42      132066
Name: count, dtype: int64

Top 10 Destination IPs in Original Dataset:
IPV4_DST_ADDR
172.31.0.2         7320073
172.31.69.25       1169767
172.31.69.28       1042284
169.254.169.254     645849
18.219.211.138      207707
0.0.0.0             108126
72.21.91.29         103554
172.31.64.83         93458
172.31.64.103        69017
178.255.83.1         58566
Name: count, dtype: int64

Unique Source IPs and Count:
181876

Unique Destination IPs and Count:
29036


# 4. Client's data construction - by departments
## 4.1 Data Cleaning

In [3]:
# Filter attack windows
ATTACK_WINDOWS_MS = [
    # (start_ms, end_ms, description)
    # Wed 14-02-2018
    (pd.Timestamp("2018-02-14 10:32").value // 1_000_000, pd.Timestamp("2018-02-14 12:09").value // 1_000_000, "FTP-BruteForce"),
    (pd.Timestamp("2018-02-14 14:01").value // 1_000_000, pd.Timestamp("2018-02-14 15:31").value // 1_000_000, "SSH-BruteForce"),
    # Thu 15-02-2018
    (pd.Timestamp("2018-02-15 09:26").value // 1_000_000, pd.Timestamp("2018-02-15 10:09").value // 1_000_000, "DoS-GoldenEye"),
    (pd.Timestamp("2018-02-15 10:59").value // 1_000_000, pd.Timestamp("2018-02-15 11:40").value // 1_000_000, "DoS-Slowloris"),
    # Fri 16-02-2018
    (pd.Timestamp("2018-02-16 10:12").value // 1_000_000, pd.Timestamp("2018-02-16 11:08").value // 1_000_000, "DoS-SlowHTTPTest"),
    (pd.Timestamp("2018-02-16 13:45").value // 1_000_000, pd.Timestamp("2018-02-16 14:19").value // 1_000_000, "DoS-Hulk"),
    # Tue 20-02-2018
    (pd.Timestamp("2018-02-20 10:12").value // 1_000_000, pd.Timestamp("2018-02-20 11:17").value // 1_000_000, "DDoS-LOIC-HTTP"),
    (pd.Timestamp("2018-02-20 13:13").value // 1_000_000, pd.Timestamp("2018-02-20 13:32").value // 1_000_000, "DDoS-LOIC-UDP"),
    # Wed 21-02-2018
    (pd.Timestamp("2018-02-21 10:09").value // 1_000_000, pd.Timestamp("2018-02-21 10:43").value // 1_000_000, "DDoS-LOIC-UDP-2"),
    (pd.Timestamp("2018-02-21 14:05").value // 1_000_000, pd.Timestamp("2018-02-21 15:05").value // 1_000_000, "DDoS-HOIC"),
    # Thu 22-02-2018
    (pd.Timestamp("2018-02-22 10:17").value // 1_000_000, pd.Timestamp("2018-02-22 11:24").value // 1_000_000, "BruteForce-Web"),
    (pd.Timestamp("2018-02-22 13:50").value // 1_000_000, pd.Timestamp("2018-02-22 14:29").value // 1_000_000, "BruteForce-XSS"),
    (pd.Timestamp("2018-02-22 16:15").value // 1_000_000, pd.Timestamp("2018-02-22 16:29").value // 1_000_000, "SQLi"),
    # Fri 23-02-2018
    (pd.Timestamp("2018-02-23 10:03").value // 1_000_000, pd.Timestamp("2018-02-23 11:03").value // 1_000_000, "BruteForce-Web-2"),
    (pd.Timestamp("2018-02-23 13:00").value // 1_000_000, pd.Timestamp("2018-02-23 14:10").value // 1_000_000, "BruteForce-XSS-2"),
    (pd.Timestamp("2018-02-23 15:05").value // 1_000_000, pd.Timestamp("2018-02-23 15:18").value // 1_000_000, "SQLi-2"),
    # Wed 28-02-2018
    (pd.Timestamp("2018-02-28 10:50").value // 1_000_000, pd.Timestamp("2018-02-28 12:05").value // 1_000_000, "Infiltration-1"),
    (pd.Timestamp("2018-02-28 13:42").value // 1_000_000, pd.Timestamp("2018-02-28 14:40").value // 1_000_000, "Infiltration-2"),
    # Thu 01-03-2018
    (pd.Timestamp("2018-03-01 09:57").value // 1_000_000, pd.Timestamp("2018-03-01 10:55").value // 1_000_000, "Infiltration-3"),
    (pd.Timestamp("2018-03-01 14:00").value // 1_000_000, pd.Timestamp("2018-03-01 15:37").value // 1_000_000, "Infiltration-4"),
    # Fri 02-03-2018
    (pd.Timestamp("2018-03-02 10:11").value // 1_000_000, pd.Timestamp("2018-03-02 11:34").value // 1_000_000, "Bot-1"),
    (pd.Timestamp("2018-03-02 14:24").value // 1_000_000, pd.Timestamp("2018-03-02 15:55").value // 1_000_000, "Bot-2"),
]

attack_mask = pd.Series(False, index=df.index)
for start_ms, end_ms, _ in ATTACK_WINDOWS_MS:
    in_window = (df["FLOW_START_MILLISECONDS"] >= start_ms) & (df["FLOW_START_MILLISECONDS"] <= end_ms)
    attack_mask |= in_window

df_benign = df[~attack_mask].copy()
print(f"Rows after removing attack windows: {len(df_benign):,}  (removed {attack_mask.sum():,})")

Rows after removing attack windows: 17,646,133  (removed 2,469,396)


In [4]:
# Known attacker IPs (exclude from topology analysis)
ATTACKER_IPS = {
    "172.31.70.4", "172.31.70.6", "172.31.70.46", "172.31.70.8",
    "172.31.70.23", "172.31.70.16",
    "18.218.115.60", "18.219.9.1", "18.219.32.43", "18.218.55.126",
    "52.14.136.135", "18.219.5.43", "18.216.200.189", "18.218.229.235",
    "18.218.11.51", "18.216.24.42", "13.58.225.34", "18.219.211.138",
    "13.58.98.64", "18.221.219.4",
}

# Internal subnet prefix
INTERNAL_PREFIX = "172.31."

def is_internal(ip: str) -> bool:
    return ip.startswith(INTERNAL_PREFIX)


# Restrict to internal to internal flows
mask_internal = (
    df_benign["IPV4_SRC_ADDR"].apply(is_internal) &
    df_benign["IPV4_DST_ADDR"].apply(is_internal) &
    ~df_benign["IPV4_SRC_ADDR"].isin(ATTACKER_IPS) &
    ~df_benign["IPV4_DST_ADDR"].isin(ATTACKER_IPS)
)
df_internal = df_benign[mask_internal]
print(f"Internal benign flows: {len(df_internal):,}")

Internal benign flows: 6,600,723


## 4.2 Unique IPs subnetworks for 172.31.

In [5]:
# Find the unique internal subnets
def extract_subnet(ip: str) -> str:
    return ".".join(ip.split(".")[:3]) + ".0/24"
df_internal["SRC_SUBNET"] = df_internal["IPV4_SRC_ADDR"].apply(extract_subnet)
df_internal["DST_SUBNET"] = df_internal["IPV4_DST_ADDR"].apply(extract_subnet)
unique_subnets = pd.unique(df_internal[["SRC_SUBNET", "DST_SUBNET"]].values.ravel())
print("Unique internal subnets:")
for subnet in unique_subnets:
    print(subnet)

Unique internal subnets:
172.31.0.0/24
172.31.66.0/24
172.31.65.0/24
172.31.67.0/24
172.31.69.0/24
172.31.64.0/24
172.31.68.0/24
172.31.230.0/24


## 4.4 Subnets based partition

In [6]:
SUBNET_TO_DEPT = {
    "172.31.64": "dep1",  
    "172.31.65": "dep2",
    "172.31.66": "dep3",
    "172.31.67": "dep4",
    "172.31.68": "dep5",
    "172.31.69": "servers", # Confirmed by known server IPs
}

def ip_to_subnet(ip: str) -> str:
    parts = ip.split(".")
    if len(parts) >= 3:
        return f"{parts[0]}.{parts[1]}.{parts[2]}"
    return "unknown"

def assign_department(row) -> str:
    src_subnet = ip_to_subnet(row["IPV4_SRC_ADDR"])
    dst_subnet = ip_to_subnet(row["IPV4_DST_ADDR"])
    if src_subnet in SUBNET_TO_DEPT:
        return SUBNET_TO_DEPT[src_subnet]
    if dst_subnet in SUBNET_TO_DEPT:
        return SUBNET_TO_DEPT[dst_subnet]
    return "External"


# Assign departments to full dataset
src_subnet = df["IPV4_SRC_ADDR"].str.extract(r"^(\d+\.\d+\.\d+)")[0]
dst_subnet = df["IPV4_DST_ADDR"].str.extract(r"^(\d+\.\d+\.\d+)")[0]
df["Department"] = src_subnet.map(SUBNET_TO_DEPT).fillna(dst_subnet.map(SUBNET_TO_DEPT)).fillna("External")
print(df["Department"].value_counts())

Department
dep1        5122142
dep3        3874680
dep2        3762566
servers     3599637
dep4        2646598
dep5        1001563
External     108343
Name: count, dtype: int64


In [7]:
# Join dep1-4 into a single client 
df["Department"] = df["Department"].replace({"dep1": "client0", "dep2": "client0", "dep3": "client0", "dep4": "client0", "dep5": "client1", "servers": "client2"})

print(df["Department"].value_counts())

Department
client0     15405986
client2      3599637
client1      1001563
External      108343
Name: count, dtype: int64


## 4.5 Data export

In [8]:
import fastparquet
PERCENT_ROWS = 0.2

curr_path = os.getcwd()
save_dir = os.path.join(curr_path, "fed_clients")
save_dir = save_dir.replace("data_analysis", "datasets")
os.makedirs(save_dir, exist_ok=True)

for dept, group in df.groupby("Department"):
    if dept == "External":
        print(f"Skipping External department with {len(group):,} rows")
        continue
    group = group.drop(columns=["Department"])
    group_size = len(group)
    if PERCENT_ROWS is not None:
        sample_size = int(group_size * PERCENT_ROWS)
        group = group.head(sample_size)
        print(f"{dept}: {group_size:,} rows → sampling {sample_size:,} rows")
    curr_dir = os.path.join(save_dir, dept)
    os.makedirs(curr_dir, exist_ok=True)
    path = os.path.join(curr_dir, f"{dept}.parquet")
    group.to_parquet(path, index=False)
    print(f"  {dept}: {len(group):,} rows → {path}")

del df["Department"]

Skipping External department with 108,343 rows
client0: 15,405,986 rows → sampling 3,081,197 rows
  client0: 3,081,197 rows → /home/carandp/FL-NIDS/datasets/fed_clients/client0/client0.parquet
client1: 1,001,563 rows → sampling 200,312 rows
  client1: 200,312 rows → /home/carandp/FL-NIDS/datasets/fed_clients/client1/client1.parquet
client2: 3,599,637 rows → sampling 719,927 rows
  client2: 719,927 rows → /home/carandp/FL-NIDS/datasets/fed_clients/client2/client2.parquet


# 5. Client's data construction - by days/attacks
## 5.1 Clients by attack/day generation

In [3]:
DAY_TO_CLIENT = {
    # client0: Volumetric / credential attacks
    "2018-02-14": "client0",   # FTP+SSH BruteForce
    "2018-02-20": "client0",   # DDoS LOIC
    "2018-03-02": "client0",   # Botnet

    # client1: Denial-of-Service
    "2018-02-15": "client1",   # DoS GoldenEye + Slowloris
    "2018-02-16": "client1",   # DoS SlowHTTPTest + Hulk
    "2018-02-21": "client1",   # DDoS HOIC

    # client2: Application-layer / stealthy attacks
    "2018-02-22": "client2",   # Web attacks day 1
    "2018-02-23": "client2",   # Web attacks day 2
    "2018-02-28": "client2",   # Infiltration day 1
    "2018-03-01": "client2",   # Infiltration day 2
}


df["date_str"] = pd.to_datetime(df["FLOW_START_MILLISECONDS"], unit="ms").dt.strftime("%Y-%m-%d")
df["client"] = df["date_str"].map(DAY_TO_CLIENT)

# Days with no attacks map to benign — distribute them round-robin
benign_days = df[df["client"].isna()]["date_str"].unique()
print("Benign-only days to distribute:", sorted(benign_days))

benign_day_assignment = {
    day: f"client{i % 3}" 
    for i, day in enumerate(sorted(benign_days))
}

null_mask = df["client"].isna()
df.loc[null_mask, "client"] = (
    df.loc[null_mask, "date_str"]
    .map(benign_day_assignment)
    .fillna("client0")
)

print("\nClass distribution per client:")
print(
    df.groupby("client")["Label"]
    .value_counts(normalize=True)
    .rename("ratio")
    .reset_index()
    .pivot(index="client", columns="Label", values="ratio")
    .round(4)
)

print("\nRow counts per client:")
print(df.groupby("client").size())

Benign-only days to distribute: ['2018-02-27']

Class distribution per client:
Label         0       1
client                 
client0  0.8153  0.1847
client1  0.7828  0.2172
client2  0.9766  0.0234

Row counts per client:
client
client0    5799886
client1    6162253
client2    8153390
dtype: int64


In [ ]:
SAMPLE_FRACTION = 0.25

dfs = []
for client_name, group in df.groupby("client"):
    sampled = (
        group.groupby("Label", group_keys=False)
        .sample(frac=SAMPLE_FRACTION, random_state=42)
    )
    dfs.append(sampled)
    print(f"{client_name}: {len(sampled):,} rows")

df_sampled = pd.concat(dfs, ignore_index=True)

client0: 2,319,954 rows
client1: 2,464,901 rows
client2: 3,261,356 rows


In [8]:
df_sampled.head(1)

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack,date_str,client
0,1519150206435,1519150206500,172.31.66.11,52575,74.125.28.99,443,6,91,237,4,40,1,25,25,17,65,65,0,128,128,86,40,40,86,0,3,0,0,0,0,28727,4848,5,0,0,0,0,258,187,0,0,0,0,0,0,0,63,21,29,0,0,0,0,0,Benign,2018-02-20,client0


In [9]:
curr_path = os.getcwd()
save_dir = os.path.join(curr_path, "fed_clients")
save_dir = save_dir.replace("data_analysis", "datasets")
os.makedirs(save_dir, exist_ok=True)

for client_name, group in df_sampled.groupby("client"):
    curr_dir = os.path.join(save_dir, client_name)
    os.makedirs(curr_dir, exist_ok=True)
    path = os.path.join(curr_dir, f"{client_name}.parquet")
    group.to_parquet(path, index=False)
    group.drop(columns=["date_str", "client"]).to_parquet(path, index=False)
    print(f"Saved {path}: {len(group):,} rows")

Saved /home/carandp/FL-NIDS/datasets/fed_clients/client0/client0.parquet: 2,319,954 rows
Saved /home/carandp/FL-NIDS/datasets/fed_clients/client1/client1.parquet: 2,464,901 rows
Saved /home/carandp/FL-NIDS/datasets/fed_clients/client2/client2.parquet: 3,261,356 rows


## 5.2 Centralized csv generation

In [ ]:
curr_path = os.getcwd()
save_dir = curr_path.replace("data_analysis", "centralized")
save_dir = os.path.join(save_dir, "datasets")
save_dir = os.path.join(save_dir, "NF-CSE-CIC-IDS2018-v3")
os.makedirs(save_dir, exist_ok=True)
centralized_path = os.path.join(save_dir, "NF-CSE-CIC-IDS2018-v3.csv")
df_sampled.drop(columns=["date_str", "client"]).to_csv(centralized_path, index=False)
print(f"Saved centralized subset: {centralized_path} with {len(df_sampled):,} rows")

Saved centralized subset: /home/carandp/FL-NIDS/centralized/datasets/NF-CSE-CIC-IDS2018-v3/NF-CSE-CIC-IDS2018-v3.csv with 8,046,211 rows


: 

# 6. Client's data construction - by servers
## 6.1 Servers identification

In [ ]:
del df["date_str"]
del df["client"]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20115529 entries, 0 to 20115528
Data columns (total 55 columns):
 #   Column                       Dtype
---  ------                       -----
 0   FLOW_START_MILLISECONDS      int64
 1   FLOW_END_MILLISECONDS        int64
 2   IPV4_SRC_ADDR                str  
 3   L4_SRC_PORT                  int32
 4   IPV4_DST_ADDR                str  
 5   L4_DST_PORT                  int32
 6   PROTOCOL                     int32
 7   L7_PROTO                     int32
 8   IN_BYTES                     int32
 9   IN_PKTS                      int32
 10  OUT_BYTES                    int32
 11  OUT_PKTS                     int32
 12  TCP_FLAGS                    int32
 13  CLIENT_TCP_FLAGS             int32
 14  SERVER_TCP_FLAGS             int32
 15  FLOW_DURATION_MILLISECONDS   int32
 16  DURATION_IN                  int32
 17  DURATION_OUT                 int32
 18  MIN_TTL                      int32
 19  MAX_TTL                      int32
 20  LONGEST_FLO

In [25]:
# Explore all server IPs with value "172.31.69"
src_unique = df[df['IPV4_SRC_ADDR'].str.startswith('172.31.69')]['IPV4_SRC_ADDR'].unique()
dst_unique = df[df['IPV4_DST_ADDR'].str.startswith('172.31.69')]['IPV4_DST_ADDR'].unique()
joined_unique = np.union1d(src_unique, dst_unique)
joined_unique.sort()
print(f"Unique server IPs starting with '172.31.69': {joined_unique}")
print(f"Total unique server IPs starting with '172.31.69': {len(joined_unique)}")

Unique server IPs starting with '172.31.69': ['172.31.69.1' '172.31.69.10' '172.31.69.11' '172.31.69.12' '172.31.69.13'
 '172.31.69.14' '172.31.69.15' '172.31.69.16' '172.31.69.17'
 '172.31.69.18' '172.31.69.19' '172.31.69.20' '172.31.69.21'
 '172.31.69.22' '172.31.69.23' '172.31.69.24' '172.31.69.25'
 '172.31.69.26' '172.31.69.27' '172.31.69.28' '172.31.69.29'
 '172.31.69.30' '172.31.69.31' '172.31.69.4' '172.31.69.5' '172.31.69.6'
 '172.31.69.7' '172.31.69.8' '172.31.69.9']
Total unique server IPs starting with '172.31.69': 29


In [29]:
# Find most common used L4_SRC_PORT and L4_DST_PORT for the server IPs starting with "172.31.69"
server_flows = df[
    df['IPV4_SRC_ADDR'].str.startswith('172.31.69') |
    df['IPV4_DST_ADDR'].str.startswith('172.31.69')
]
top_src_ports = server_flows['L4_SRC_PORT'].value_counts().head(10)
top_dst_ports = server_flows['L4_DST_PORT'].value_counts().head(40)
print("Top 10 L4_SRC_PORTs for server IPs starting with '172.31.69':")
print(top_src_ports)
print("\nTop 40 L4_DST_PORTs for server IPs starting with '172.31.69':")
print(top_dst_ports)

Top 10 L4_SRC_PORTs for server IPs starting with '172.31.69':
L4_SRC_PORT
123      5937
0        4649
443      4585
3389     4116
49672    4072
68       3928
63875    3164
59149    3141
43299    3001
46131    3000
Name: count, dtype: int64

Top 40 L4_DST_PORTs for server IPs starting with '172.31.69':
L4_DST_PORT
80       1557502
53        647324
21        492574
8080      208400
22        193536
443       174481
3389      105737
445        47220
23          6788
123         6480
0           4649
500         4369
67          3931
135         3730
138         1835
8545        1407
5060        1314
139         1180
5355         865
2323         657
1433         611
81           481
137          468
1900         433
5555         418
3306         375
52869        284
3390         252
5900         243
1            242
389          234
2222         224
9000         210
161          202
1434         163
111          162
8081         156
17           153
222          153
5903         152
Name:

We can see 
* App Server ports: 80 (HTTP), 8080 (HTTP), 443 (HTTPS)
* File Server ports: 21 (FTP), 22 (SFTP), 138 (NetBIOS), 139 (NetBIOS), 445 (SMB)
* Active Directory (AD) ports: 53 (DNS), 135 (RPC), 3389 (RDP)
* Email Server ports: NO 25 port
* ADD ports: 500 (VPN), 5060 (VoIP), 3306 (MySQL), 1433 (Microsoft SQL Server) 

In [3]:
app_ports = {80, 8080, 443}
file_ports = {21, 22, 138, 139, 445, 3389}
ad_ports = {53, 135, 500, 5060, 3306, 1433}

def filter_by_ports(dataframe, ports):
    return dataframe[
        dataframe["L4_SRC_PORT"].isin(ports) |
        dataframe["L4_DST_PORT"].isin(ports)
    ]

df_app = filter_by_ports(df, app_ports)
df_file = filter_by_ports(
    df[~df.index.isin(df_app.index)],
    file_ports
)

df_ad = filter_by_ports(
    df[
        ~df.index.isin(df_app.index) &
        ~df.index.isin(df_file.index)
    ],
    ad_ports
)

df_other = df[
    ~df.index.isin(df_app.index) &
    ~df.index.isin(df_file.index) &
    ~df.index.isin(df_ad.index)
]

print(f"App-related flows: {len(df_app):,}")
print(f"File-related flows: {len(df_file):,}")
print(f"AD-related flows: {len(df_ad):,}")
print(f"Other server flows: {len(df_other):,}")
print(f"Total flows: {len(df):,} (should equal sum of above)")

App-related flows: 6,045,943
File-related flows: 5,578,635
AD-related flows: 7,385,573
Other server flows: 1,105,378
Total flows: 20,115,529 (should equal sum of above)


In [4]:
# Check the Label distribution in each category
print("\nLabel distribution in App-related flows:")
print(df_app["Label"].value_counts(normalize=True).round(4))
print("\nLabel distribution in File-related flows:")
print(df_file["Label"].value_counts(normalize=True).round(4))
print("\nLabel distribution in AD-related flows:")
print(df_ad["Label"].value_counts(normalize=True).round(4))
print("\nLabel distribution in Other server flows:")
print(df_other["Label"].value_counts(normalize=True).round(4))


Label distribution in App-related flows:
Label
0    0.711
1    0.289
Name: proportion, dtype: float64

Label distribution in File-related flows:
Label
0    0.8746
1    0.1254
Name: proportion, dtype: float64

Label distribution in AD-related flows:
Label
0    0.9854
1    0.0146
Name: proportion, dtype: float64

Label distribution in Other server flows:
Label
0    0.9581
1    0.0419
Name: proportion, dtype: float64


In [5]:
SAMPLE_FRACTION = 0.25

dfs = []
for category_name, group in [("app", df_app), ("file", df_file), ("ad", df_ad)]:
    pieces = []
    for label_val, label_group in group.groupby("Label"):
        if category_name == "ad" and label_val == 1:
            piece = label_group.sort_values("FLOW_START_MILLISECONDS")
        else:
            n = max(1, int(len(label_group) * SAMPLE_FRACTION))
            piece = label_group.sort_values("FLOW_START_MILLISECONDS").head(n)
        pieces.append(piece)
    
    sampled = pd.concat(pieces, ignore_index=True)
    dfs.append(sampled)
    print(f"{category_name}: {len(sampled):,} rows")

df_sampled = pd.concat(dfs, ignore_index=True)

app: 1,511,485 rows
file: 1,394,658 rows
ad: 1,926,995 rows


In [6]:
# Recheck the Label distribution in each category after sampling
print("\nLabel distribution in App-related flows after sampling:")
print(dfs[0]["Label"].value_counts(normalize=True).round(4))
print("\nLabel distribution in File-related flows after sampling:") 
print(dfs[1]["Label"].value_counts(normalize=True).round(4))
print("\nLabel distribution in AD-related flows after sampling:")
print(dfs[2]["Label"].value_counts(normalize=True).round(4))

# Check final df_sampled distribution
print("\nOverall Label distribution in sampled dataset:")
print(df_sampled["Label"].value_counts(normalize=True).round(4))


Label distribution in App-related flows after sampling:
Label
0    0.711
1    0.289
Name: proportion, dtype: float64

Label distribution in File-related flows after sampling:
Label
0    0.8746
1    0.1254
Name: proportion, dtype: float64

Label distribution in AD-related flows after sampling:
Label
0    0.9442
1    0.0558
Name: proportion, dtype: float64

Overall Label distribution in sampled dataset:
Label
0    0.8512
1    0.1488
Name: proportion, dtype: float64


In [7]:
curr_path = os.getcwd()
save_dir = os.path.join(curr_path, "fed_clients")
save_dir = save_dir.replace("data_analysis", "datasets")
os.makedirs(save_dir, exist_ok=True)

for category_name, group in zip(["client0", "client1", "client2"], dfs):
    curr_dir = os.path.join(save_dir, category_name)
    os.makedirs(curr_dir, exist_ok=True)
    path = os.path.join(curr_dir, f"{category_name}.parquet")
    group.to_parquet(path, index=False)
    print(f"Saved {path}: {len(group):,} rows")

Saved /home/carandp/FL-NIDS/datasets/fed_clients/client0/client0.parquet: 1,511,485 rows
Saved /home/carandp/FL-NIDS/datasets/fed_clients/client1/client1.parquet: 1,394,658 rows
Saved /home/carandp/FL-NIDS/datasets/fed_clients/client2/client2.parquet: 1,926,995 rows


In [8]:
# Centralized csv save
curr_path = os.getcwd()
save_dir = curr_path.replace("data_analysis", "centralized")
save_dir = os.path.join(save_dir, "datasets")
save_dir = os.path.join(save_dir, "NF-CSE-CIC-IDS2018-v3")
os.makedirs(save_dir, exist_ok=True)
centralized_path = os.path.join(save_dir, "NF-CSE-CIC-IDS2018-v3.csv")
df_sampled.to_csv(centralized_path, index=False)
print(f"Saved centralized subset: {centralized_path} with {len(df_sampled):,} rows")

Saved centralized subset: /home/carandp/FL-NIDS/centralized/datasets/NF-CSE-CIC-IDS2018-v3/NF-CSE-CIC-IDS2018-v3.csv with 4,833,138 rows
